In [ ]:
import sys
import os

# Use absolute paths to ensure imports work from anywhere
pipeline_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline"
eda_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline\eda"

sys.path.insert(0, eda_dir)
sys.path.insert(0, pipeline_dir)

import polars as pl
import numpy as np
import pandas as pd
from config import *
from eda_base import *
from eda_stats import *

# Raw Employment Spell Data (S1 Output)
All employment spells with computed flags (49 flag columns, not filters)

In [ ]:
# Load sample year to understand structure
raw_2010 = pl.scan_parquet(os.path.join(OUT_DIR, 'raw', 'raw_2010.parquet')).collect()
print(f"Shape: {raw_2010.shape}")
print(f"\nColumns ({len(raw_2010.columns)}): {raw_2010.columns}")


In [ ]:
# Time dimension: row counts across all years
year_counts = []
for year in PROJECT_YEARS:
    try:
        path = os.path.join(OUT_DIR, 'raw', f'raw_{year}.parquet')
        n = pl.scan_parquet(path).select(pl.len()).collect()["len"][0]
        year_counts.append({"year": year, "n_spells": n})
    except:
        pass
        
df_counts = pl.DataFrame(year_counts)
print("Employment spells by year:")
print(df_counts)


In [ ]:
# Missing values: worker identifiers
missing_cols = ['ident_all', 'ident_s', 'siren', 'siret', 'comt']
missing_stats = {}
for c in missing_cols:
    if c in raw_2010.columns:
        null_count = raw_2010.select(pl.col(c).is_null().sum()).to_series()[0]
        pct = 100 * null_count / len(raw_2010)


In [ ]:
# Worker panel structure: workers per year
workers_per_year = raw_2010.group_by('an').agg(pl.col('ident_all').n_unique().alias('n_workers')).sort('an')
print("Unique workers per year (2010):")
print(workers_per_year)

# Average spells per worker
spells_per_worker = raw_2010.select([
    (pl.len() / pl.col('ident_all').n_unique()).alias('avg_spells_per_worker')
])


In [ ]:
# Flag columns overview: count of 49 computed flags
flag_cols = [c for c in raw_2010.columns if c.startswith('flag_')]
print(f"Total flag columns: {len(flag_cols)}")
print(f"\nFlag columns: {flag_cols[:10]}...") if len(flag_cols) > 10 else print(f"\nFlag columns: {flag_cols}")

# Flag coverage: how many spells have each flag set
print("\nFlag coverage (share of spells):")
for col in flag_cols[:10]:  # first 10 flags
    if col in raw_2010.columns:
        coverage = raw_2010.select(pl.col(col).sum()).to_series()[0] / len(raw_2010)


In [ ]:
# Wage statistics
wage_cols = [c for c in raw_2010.columns if 'brut' in c or 'wage' in c or 'salaire' in c]
print(f"Wage-related columns: {wage_cols}")

if 's_brut' in raw_2010.columns:
    wage_stats = raw_2010.select([
        pl.col('s_brut').min().alias('min'),
        pl.col('s_brut').max().alias('max'),
        pl.col('s_brut').mean().alias('mean'),
        pl.col('s_brut').median().alias('median'),
        (pl.col('s_brut') > 0).sum().alias('n_nonzero')
    ])
    print("\nGross wage statistics (s_brut):")


In [ ]:
# Temporal consistency: spell dates
if 'date_debut' in raw_2010.columns and 'date_fin' in raw_2010.columns:
    date_check = raw_2010.select([
        (pl.col('date_debut') <= pl.col('date_fin')).sum().alias('valid_spells'),
        (pl.col('date_debut') > pl.col('date_fin')).sum().alias('invalid_spells')
    ])
    print("Spell date validity check:")
    print(date_check)
    pct_valid = date_check['valid_spells'][0] / (date_check['valid_spells'][0] + date_check['invalid_spells'][0]) * 100


In [ ]:
# Geographic distribution: top communes
spatial_summary = raw_2010.group_by('comt').agg(pl.len().alias('n_spells')).sort('n_spells', descending=True).head(10)
print("Top 10 communes by employment spells (2010):")
